In [ ]:
# %pip install seleniumbase

   ---------------------------------------- 0.0/659.5 kB ? eta -:--:--
   ---------------------------------------- 659.5/659.5 kB 26.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------- ----- 8.4/9.6 MB 40.0 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 35.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 33.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 23.2 MB/s  0:00:00

   - --------------------------------------  2/54 [sortedcontainers]
   -- -------------------------------------  4/54 [zipp]
   -- -------------------------------------  4/54 [zipp]
   --- ------------------------------------  5/54 [wsproto]
   ---- -----------------------------------  6/54 [websockets]
   ---- ------------------------------

In [2]:
from seleniumbase import SB
import pandas as pd
import re
import time
import os
from datetime import datetime, timezone, timedelta
import random
import urllib.parse

BASE_DOMAIN = "https://opga037.com"
BOARD_URL_TEMPLATE = (
    "https://opga037.com/bbs/board.php?"
    "bo_table=op_partner_posting"
    "&cat=1"
    "&addrName={}"
    "&biz=0"
    "&bizName=%EC%A0%84%EC%B2%B4"
)

LOGIN_ID = "ccl12345"
LOGIN_PW = "ccl12345"

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")

def now_kst():
    kst = timezone(timedelta(hours=9))
    return datetime.now(kst).strftime("%Y-%m-%d %H:%M:%S")

In [4]:
with SB(uc=True, headed=True) as sb:

    # 1. 메인 접속 (Cloudflare 대응)
    sb.open(BASE_DOMAIN)
    sb.sleep(5)

    # 보안 페이지면 수동 확인 대기
    if "보안 확인" in sb.get_page_source():
        print("Cloudflare 확인 필요. 체크 후 Enter.")
        input()
        sb.sleep(3)

    # 2. 로그인 페이지 이동
    sb.open(BASE_DOMAIN + "/bbs/login.php")
    sb.sleep(2)

    # 3. 로그인 입력
    sb.type("#login_id", LOGIN_ID)
    sb.type("#login_pw", LOGIN_PW)
    sb.click("button[type='submit']")
    sb.sleep(5)

    print(sb.driver.get_cookies())
    print("로그인 후 URL:", sb.get_current_url())
    print("로그아웃 존재:", "로그아웃" in sb.get_page_source())

RuntimeError: Cannot run the event loop while another loop is running

In [ ]:
def move_area(sb, area_name):
    encoded = urllib.parse.quote(area_name)
    url = BOARD_URL_TEMPLATE.format(encoded)
    sb.open(url)
    sb.sleep(3)
    print("이동 완료:", sb.get_current_url())

In [ ]:
def scroll_to_bottom(sb, pause=1.5, max_round=200):
    last_height = sb.execute_script("return document.body.scrollHeight")

    for i in range(max_round):
        sb.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        sb.sleep(pause)

        new_height = sb.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            print("스크롤 종료")
            break
        last_height = new_height

In [ ]:
def parse_store_name(text):
    m = re.search(r"\[(.*?)\]", text)
    if not m:
        return None

    inside = m.group(1).strip()
    if "-" in inside:
        return inside.split("-", 1)[1].strip()
    return inside

In [ ]:
def collect_detail_urls(sb):
    links = sb.find_elements("a[href*='wr_id=']")
    urls = []

    for a in links:
        href = a.get_attribute("href")
        if href and "wr_id=" in href:
            urls.append(href)

    return list(dict.fromkeys(urls))

In [ ]:
def extract_external_id(url):
    m = re.search(r"wr_id=(\d+)", url)
    return m.group(1) if m else None

def extract_phone(text):
    m = re.search(r"(0\d{1,2}-\d{3,4}-\d{4})", text)
    return m.group(1) if m else None

def extract_category(text):
    keywords = ["스웨디시", "마사지", "안마", "건마", "1인샵", "테라피", "타이"]
    found = [k for k in keywords if k in text]
    return " | ".join(sorted(set(found))) if found else None

In [ ]:
results = []
visited_urls = set()

AREAS = ["서울-강남", "서울-비강남"]

with SB(uc=True, headed=True) as sb:

    # 1. 메인 접속 (Cloudflare 대응)
    sb.open(BASE_DOMAIN)
    sb.sleep(5)

    if "보안 확인" in sb.get_page_source():
        print("Cloudflare 확인 필요. 체크 후 Enter.")
        input()
        sb.sleep(3)

    # 2. 로그인
    sb.open(BASE_DOMAIN + "/bbs/login.php")
    sb.type("#login_id", LOGIN_ID)
    sb.type("#login_pw", LOGIN_PW)
    sb.click("button[type='submit']")
    sb.sleep(5)

    print("로그인 완료:", "로그아웃" in sb.get_page_source())

    crawled_at = now_kst()

    # 3. 지역 루프
    for area in AREAS:

        print(f"\n===== {area} 수집 시작 =====")

        move_area(sb, area)
        scroll_to_bottom(sb)

        detail_urls = collect_detail_urls(sb)
        print(f"{area} 상세 URL 수:", len(detail_urls))

        for url in detail_urls:

            if url in visited_urls:
                continue

            visited_urls.add(url)

            try:
                sb.open(url)
                sb.sleep(random.uniform(1.5, 3.5))

                page_text = sb.get_page_source()

                name = None
                title_candidates = sb.find_elements("span.list_op_head")
                if title_candidates:
                    name = parse_store_name(title_candidates[0].text)

                row = {
                    "source_site": "opga037.com",
                    "crawled_at": crawled_at,
                    "listing_url": sb.get_current_url(),
                    "detail_url": url,
                    "external_id": extract_external_id(url),
                    "name_raw": name,
                    "category_raw": extract_category(page_text),
                    "phone_number": extract_phone(page_text),
                    "area": area
                }

                results.append(row)

            except Exception as e:
                print("에러 발생:", url)
                continue

df = pd.DataFrame(results)
print("총 수집 건수:", len(df))
df.head()

In [ ]:
kst = timezone(timedelta(hours=9))
ts = datetime.now(kst).strftime("%y%m%d_%H%M%S")

file_name = f"opguide_raw_{ts}.csv"
save_path = os.path.join(DATA_DIR, file_name)

df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", file_name)